<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/polars-01/cuadernos/Polars_Cuaderno04_Seleccionar_columnas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Cuaderno 4 - Seleccionar columnas con `select()`

## Antes de comenzar

En el capítulo teórico vimos que Polars no organiza la selección de datos alrededor de un índice semejante al de Pandas. En este cuaderno vamos a empezar a trabajar con la forma propia que propone Polars para construir resultados a partir de las columnas de un DataFrame.

Nos concentraremos en `select()` y en una primera aproximación a `pl.col()`. El objetivo no es desarrollar todavía todo el sistema de expresiones, sino observarlo en funcionamiento y aprender a leer algunas construcciones sencillas.

Trabajaremos nuevamente con un pequeño conjunto de ventas de una tienda escolar. Su tamaño reducido nos permitirá comparar fácilmente cada resultado con los datos originales.


## 1. Preparar el entorno y los datos

Instalamos Polars en el entorno de Colab e importamos la biblioteca utilizando la abreviatura habitual `pl`.


In [20]:
!pip install -q polars

In [21]:
import polars as pl

Creamos ahora el mismo tipo de DataFrame que venimos utilizando. De esta manera podemos concentrarnos en la selección de columnas sin introducir todavía un conjunto de datos nuevo.


In [22]:
datos = {
    "producto": [
        "Cuaderno",
        "Lapicera",
        "Mochila",
        "Calculadora",
        "Regla",
        "Carpeta",
    ],
    "categoria": [
        "Librería",
        "Librería",
        "Accesorios",
        "Tecnología",
        "Librería",
        "Librería",
    ],
    "precio": [
        3500,
        1200,
        28500,
        18500,
        900,
        4200,
    ],
    "cantidad": [
        4,
        10,
        2,
        3,
        6,
        5,
    ],
}

df = pl.DataFrame(datos)

df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


El DataFrame ejecutado tiene `shape: (6, 4)`: contiene seis filas y cuatro columnas, llamadas `producto`, `categoria`, `precio` y `cantidad`. A partir de esta tabla iremos construyendo nuevos resultados sin modificar el objeto original.


## 2. Seleccionar una sola columna

La forma más directa de utilizar `select()` consiste en indicar el nombre de una columna.


In [23]:
df.select("producto")

producto
str
"""Cuaderno"""
"""Lapicera"""
"""Mochila"""
"""Calculadora"""
"""Regla"""
"""Carpeta"""


La salida tiene `shape: (6, 1)`. Conservamos las seis filas y obtenemos un DataFrame con una sola columna, `producto`. Todavía no estamos filtrando registros; únicamente estamos decidiendo qué variable forma parte del nuevo resultado.


## 3. Seleccionar varias columnas

Podemos indicar más de un nombre dentro de `select()`.


In [24]:
df.select("producto", "precio")

producto,precio
str,i64
"""Cuaderno""",3500
"""Lapicera""",1200
"""Mochila""",28500
"""Calculadora""",18500
"""Regla""",900
"""Carpeta""",4200


La salida tiene ahora `shape: (6, 2)` y contiene únicamente `producto` y `precio`. Las columnas `categoria` y `cantidad` no forman parte de esta nueva tabla, aunque continúan presentes en `df`.


También podemos decidir el orden en que aparecerán las columnas, independientemente del orden que tengan en el DataFrame original.


In [25]:
df.select("cantidad", "producto")

cantidad,producto
i64,str
4,"""Cuaderno"""
10,"""Lapicera"""
2,"""Mochila"""
3,"""Calculadora"""
6,"""Regla"""
5,"""Carpeta"""


La nueva salida también tiene seis filas y dos columnas, pero el orden cambió: `cantidad` aparece primero y `producto` después. Esto confirma que `select()` no solo permite elegir columnas, sino también establecer explícitamente el orden en que formarán parte del resultado.


## 4. Una primera expresión con `pl.col()`

Polars también permite referirse a una columna mediante `pl.col()`.


In [26]:
pl.col("precio")

<Expr ['col("precio")'] at 0x7B7E2DE9A650>

La salida de `pl.col("precio")` no contiene los seis precios. Polars muestra un objeto `Expr`, es decir, una expresión que hace referencia a una columna llamada `precio`.

Todavía no se ha evaluado esa referencia sobre ningún DataFrame. Para obtener los valores necesitamos utilizar la expresión dentro de un contexto, como `select()`.


In [27]:
df.select(pl.col("precio"))

precio
i64
3500
1200
28500
18500
900
4200


Al colocar la expresión dentro de `df.select()`, la salida pasa a ser un DataFrame de `shape: (6, 1)` que contiene los valores `3500`, `1200`, `28500`, `18500`, `900` y `4200` de la columna `precio`.

La diferencia entre la expresión y su resultado queda así visible: `pl.col("precio")` describe una referencia; `df.select(pl.col("precio"))` la evalúa sobre `df`.


## 5. Varias columnas mediante `pl.col()`

Podemos construir una selección utilizando varias expresiones `pl.col()`.


In [28]:
df.select(
    pl.col("producto"),
    pl.col("precio"),
)

producto,precio
str,i64
"""Cuaderno""",3500
"""Lapicera""",1200
"""Mochila""",28500
"""Calculadora""",18500
"""Regla""",900
"""Carpeta""",4200


La salida vuelve a tener `shape: (6, 2)` y reproduce exactamente las columnas `producto` y `precio`. En una selección sencilla como esta, escribir dos expresiones `pl.col()` produce el mismo resultado que indicar directamente los nombres de las columnas.


También podemos indicar varios nombres dentro de una sola llamada a `pl.col()`.


In [29]:
df.select(pl.col("producto", "precio"))

producto,precio
str,i64
"""Cuaderno""",3500
"""Lapicera""",1200
"""Mochila""",28500
"""Calculadora""",18500
"""Regla""",900
"""Carpeta""",4200


Una sola expresión `pl.col("producto", "precio")` se expande sobre las dos columnas indicadas. La salida vuelve a tener seis filas y dos columnas: `producto` y `precio`. Más adelante aprovecharemos esta capacidad para trabajar con grupos de columnas de una forma más expresiva.


## 6. Seleccionar todas las columnas

Dentro del sistema de expresiones, `pl.all()` representa todas las columnas disponibles.


In [30]:
df.select(pl.all())

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida recupera el `shape: (6, 4)` original y muestra nuevamente `producto`, `categoria`, `precio` y `cantidad`. En este ejemplo `pl.all()` reproduce todo el DataFrame, aunque más adelante será especialmente útil cuando queramos combinar una selección amplia con otras transformaciones.


## 7. Excluir columnas

A partir de `pl.all()` podemos indicar columnas que queremos dejar fuera del resultado.


In [31]:
df.select(pl.all().exclude("categoria"))

producto,precio,cantidad
str,i64,i64
"""Cuaderno""",3500,4
"""Lapicera""",1200,10
"""Mochila""",28500,2
"""Calculadora""",18500,3
"""Regla""",900,6
"""Carpeta""",4200,5


La salida tiene `shape: (6, 3)`: aparecen `producto`, `precio` y `cantidad`, mientras que `categoria` quedó excluida.


In [32]:
df.select(pl.all().exclude("categoria", "cantidad"))

producto,precio
str,i64
"""Cuaderno""",3500
"""Lapicera""",1200
"""Mochila""",28500
"""Calculadora""",18500
"""Regla""",900
"""Carpeta""",4200


Al excluir también `cantidad`, la salida queda reducida a `shape: (6, 2)` y contiene solamente `producto` y `precio`. Esta forma de selección resulta conveniente cuando queremos conservar casi todas las variables y descartar únicamente unas pocas.


## 8. `select()` también puede evaluar cálculos

Hasta ahora utilizamos `select()` para construir tablas a partir de columnas existentes. Sin embargo, también puede evaluar expresiones.


In [33]:
df.select(pl.col("precio") * 2)

precio
i64
7000
2400
57000
37000
1800
8400


La salida muestra una sola columna de tipo entero con los valores `7000`, `2400`, `57000`, `37000`, `1800` y `8400`: cada precio original fue multiplicado por dos.

Esto confirma que `select()` no se limita a extraer columnas existentes. También puede evaluar una expresión y utilizar sus resultados para construir el nuevo DataFrame.


## 9. Combinar dos columnas

Podemos construir una expresión utilizando más de una columna. En este caso multiplicamos `precio` por `cantidad`.


In [34]:
df.select(pl.col("precio") * pl.col("cantidad"))

precio
i64
14000
12000
57000
55500
5400
21000


La salida contiene los valores `14000`, `12000`, `57000`, `55500`, `5400` y `21000`. Cada uno surge de multiplicar `precio` por `cantidad` en una misma fila. Por ejemplo, `Cuaderno` produce `3500 × 4 = 14000`, mientras que `Lapicera` produce `1200 × 10 = 12000`.

Observamos también que, como todavía no asignamos un nombre nuevo, la columna resultante conserva el nombre `precio`. En la siguiente operación resolveremos ese detalle con `alias()`.


## 10. Dar nombre al resultado con `alias()`

Cuando una expresión produce una columna nueva conviene asignarle un nombre que describa su significado.


In [35]:
df.select(
    (pl.col("precio") * pl.col("cantidad")).alias("importe")
)

importe
i64
14000
12000
57000
55500
5400
21000


La salida contiene ahora una columna llamada `importe`, con los mismos seis valores calculados anteriormente: `14000`, `12000`, `57000`, `55500`, `5400` y `21000`. `alias("importe")` no cambia el cálculo; asigna un nombre más representativo al resultado de la expresión.


También podemos combinar columnas originales y calculadas dentro del mismo `select()`.


In [36]:
df.select(
    "producto",
    "precio",
    "cantidad",
    (pl.col("precio") * pl.col("cantidad")).alias("importe"),
)

producto,precio,cantidad,importe
str,i64,i64,i64
"""Cuaderno""",3500,4,14000
"""Lapicera""",1200,10,12000
"""Mochila""",28500,2,57000
"""Calculadora""",18500,3,55500
"""Regla""",900,6,5400
"""Carpeta""",4200,5,21000


La salida tiene `shape: (6, 4)` y reúne `producto`, `precio`, `cantidad` e `importe`. Las tres primeras columnas proceden directamente de `df`, mientras que `importe` fue construido al multiplicar `precio` por `cantidad`.

Esta tabla muestra con claridad que `select()` puede combinar columnas existentes y resultados calculados dentro de un mismo DataFrame.


## 11. El DataFrame original no se modifica

Volvemos a observar `df` después de todas las selecciones y cálculos anteriores.


In [37]:
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida vuelve a mostrar el DataFrame original con `shape: (6, 4)` y sus cuatro columnas iniciales: `producto`, `categoria`, `precio` y `cantidad`. `importe` no fue agregado a `df`.

Todas las llamadas anteriores a `select()` produjeron nuevos resultados sin modificar el DataFrame de partida.


Si necesitamos conservar una de esas tablas para utilizarla más adelante, podemos asignarla a otra variable.


In [38]:
resumen_ventas = df.select(
    "producto",
    (pl.col("precio") * pl.col("cantidad")).alias("importe"),
)

resumen_ventas

producto,importe
str,i64
"""Cuaderno""",14000
"""Lapicera""",12000
"""Mochila""",57000
"""Calculadora""",55500
"""Regla""",5400
"""Carpeta""",21000


`resumen_ventas` tiene `shape: (6, 2)` y contiene únicamente `producto` e `importe`. Los importes calculados son `14000`, `12000`, `57000`, `55500`, `5400` y `21000`.

De esta manera conservamos el resultado que nos interesa en una variable nueva, mientras `df` mantiene intacta su estructura original.


## 12. Leer una expresión completa

Consideremos nuevamente esta construcción:

```python
df.select(
    "producto",
    (pl.col("precio") * pl.col("cantidad")).alias("importe"),
)
```

Podemos leerla de manera directa:

> A partir de `df`, construí un DataFrame que contenga `producto` y una columna llamada `importe`, calculada multiplicando `precio` por `cantidad`.

Esta forma de interpretar el código será más útil que memorizar cada elemento de manera aislada. En los próximos capítulos las expresiones se volverán más ricas, pero la idea general seguirá siendo la misma: indicar qué columnas participan, describir la operación y utilizar un contexto en el que esa descripción pueda evaluarse.


## 13. Qué conviene recordar

En este cuaderno comenzamos a trabajar con dos elementos fundamentales de Polars:

```text
select()
pl.col()
```

`select()` construye un nuevo DataFrame a partir de las columnas y expresiones que indicamos. `pl.col()` permite referirse a una columna dentro del sistema de expresiones.

También utilizamos `pl.all()`, `exclude()` y `alias()`, y comprobamos que una expresión puede hacer algo más que seleccionar una columna existente: también puede realizar cálculos y producir una nueva columna.

Todavía no hemos desarrollado el sistema de expresiones en profundidad. El propósito de este cuaderno fue empezar a utilizarlo en situaciones sencillas y observar la lógica con la que Polars construye nuevos resultados.


## Próximo paso

Todas las operaciones realizadas hasta aquí conservaron las seis filas originales. Seleccionamos columnas, cambiamos su orden y construimos cálculos, pero todavía no decidimos qué registros debían permanecer en la tabla.

En el próximo capítulo utilizaremos `filter()` para conservar únicamente las filas que cumplan una condición. Allí veremos cómo una expresión como:

```python
pl.col("precio") > 5000
```

puede utilizarse para describir qué registros queremos conservar.


---

### Autoría y atribución

Este cuaderno forma parte del material educativo desarrollado por **VintaBytes**.

📧 **Contacto:** [vintabytes@gmail.com](mailto:vintabytes@gmail.com)

🌐 **Repositorio oficial:** [github.com/VintaBytes/Ciencia-de-datos](https://github.com/VintaBytes/Ciencia-de-datos)

Se permite compartir y reutilizar este material respetando la autoría y manteniendo visible esta atribución.

---
